In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def get_summary_df(results_df):
    grouped = results_df.groupby('Alpha')
    mean_summary = grouped.mean(numeric_only=True)
    sem_summary = grouped.sem(numeric_only=True)
    ci95_lower = mean_summary - 1.96 * sem_summary
    ci95_upper = mean_summary + 1.96 * sem_summary
    ci95_lower.columns = [f"{col}_ci_lower" for col in ci95_lower.columns]
    ci95_upper.columns = [f"{col}_ci_upper" for col in ci95_upper.columns]

    summary_df = pd.concat([mean_summary, ci95_lower, ci95_upper], axis=1)
    return summary_df

In [ ]:
def relative_ci_width(summary_df, metric):
    lower = summary_df[f"{metric}_ci_lower"]
    upper = summary_df[f"{metric}_ci_upper"]
    mean = summary_df[metric]

    return ((upper - lower) / mean).rename(f"{metric}_relative_ci_width")

In [3]:
def plot_column(df, column):
    x = df.index
    y = df[column]
    y_lower = df[f"{column}_ci_lower"]
    y_upper = df[f"{column}_ci_upper"]

    plt.figure(figsize=(10, 6))
    plt.plot(x, y, marker='o', label='Mean')
    plt.fill_between(x, y_lower, y_upper, color='b', alpha=0.2, label='95% CI')

    plt.title(f"{column} with 95% Confidence Interval")
    plt.xlabel("Alpha")
    plt.ylabel(column)
    plt.legend()
    plt.grid()
    plt.show()

In [4]:
def plot_summary_df(summary_df):
    metrics = [col for col in summary_df.columns if not any(x in col for x in ["ci_lower", "ci_upper"])]

    ncols = 2
    nrows = (len(metrics) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 5 * nrows))
    axes = axes.flatten() 

    x = summary_df.index

    for i, metric in enumerate(metrics):
        y = summary_df[metric]
        y_lower = summary_df[f"{metric}_ci_lower"]
        y_upper = summary_df[f"{metric}_ci_upper"]
        
        ax = axes[i]
        ax.plot(x, y, marker='o', label='Mean')
        ax.fill_between(x, y_lower, y_upper, color='b', alpha=0.2, label='95% CI')
        ax.set_title(metric)
        ax.set_xlabel("Alpha")
        ax.set_ylabel(metric)
        ax.legend()
        ax.grid()

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle("Mean and 95% CI across Alpha values", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [5]:
def plot_column_across_levels(df1, df2, df3, column):
    summary_dfs = [df1, df2, df3]
    labels = ['Low', 'Medium', 'High']
    col = column

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for i, (df, label) in enumerate(zip(summary_dfs, labels)):
        x = df.index
        y = df[col]
        y_lower = df[f"{col}_ci_lower"]
        y_upper = df[f"{col}_ci_upper"]

        ax = axes[i]
        ax.plot(x, y, marker='o', label='Mean')
        ax.fill_between(x, y_lower, y_upper, color='b', alpha=0.2, label='95% CI')
        
        ax.set_title(f"{label} {col}")
        if i == 0:
            ax.set_ylabel(col)
        ax.grid(True)
        ax.legend()
    
    fig.suptitle(f"{col} across Levels", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [6]:
def plot_summaries_across_levels(df1, df2, df3):
    summaries = [df1, df2, df3]
    titles = ["Low", "Medium", "High"]

    metrics = [col for col in df1.columns if not any(x in col for x in ["ci_lower", "ci_upper"])]

    nrows, ncols = len(metrics), len(summaries) 
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 5 * nrows))
    if nrows == 1:
        axes = [axes]

    for row_idx, metric in enumerate(metrics):
        for col_idx, (summary, title) in enumerate(zip(summaries, titles)):
            ax = axes[row_idx][col_idx] if nrows > 1 else axes[col_idx]

            x = summary.index
            y = summary[metric]
            y_lower = summary[f"{metric}_ci_lower"]
            y_upper = summary[f"{metric}_ci_upper"]

            ax.plot(x, y, marker='o', label='Mean')
            ax.fill_between(x, y_lower, y_upper, color='b', alpha=0.2, label='95% CI')

            ax.set_title(f"{title} - {metric}")
            ax.set_xlabel("Alpha")
            ax.set_ylabel(metric)
            ax.legend()
            ax.grid()

    plt.tight_layout()
    plt.show()

### Subspecialties and Lengths